# 🎯 [DAT06_M01] 군수 월간 결산 항목(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률) 데이터 구조 분석
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT06/dat06_m01_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT06_M01 |
| **미션 주제** | 군수 월간 결산 항목(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률) 데이터 구조 분석 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
미국 국방재무회계국(DFAS)은 반복적인 재무·정산 업무에 업무 자동화(RPA)를 도입해 사람이 하던 취합과 점검을 줄이는 사업을 추진해 왔다. 이 공개 사례의 학습 초점은 반복 업무 분해, 입력과 트리거, 자동 처리 순서, 오류 복구, 실행 로그, 시간 절감이다. 본 프로젝트는 이를 바탕으로 교육용 가상 부대 24개의 월간 결산 자료(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)를 코드로 만들고, 파일 대신 메모리 안의 가상 문서와 표로 모의한다. 학습자는 업무 분해, 다형식 파싱, 검증 룰 엔진, 이상 원인 추천, 일관성 검증, 승인·반려 대시보드, 3단계 사람 결재선, 대량 배치 시험, AAR 보정, 최종 통합을 차례로 수행한다. 최종 목표는 월간 정산·보고·취합을 한 번에 실행하되 발간은 반드시 사람이 결재하도록 하는 원클릭 자동화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> 💡 **[산출물(Deliverables) 디렉토리 경로 및 실습 안내]**
> - **아키텍처 관점의 표준 산출물:** 과제 명세에 표기된 src/connectors/..., docs/..., dist/... 등의 경로는 본 솔루션이 실무 배포 환경에서 갖추게 되는 **소프트웨어 표준 아키텍처 구조**를 의미합니다.
> - **노트북 중심 실습:** 별도의 폴더나 외부 파이썬 파일(.py)을 수작업으로 생성하실 필요가 없습니다. 본 노트북의 각 미션별 실습 셀을 AI 코파일럿과 함께 완성하고 검증하면 해당 산출물 요건이 자동으로 충족됩니다.
> - **미션 10 최종 통합 패키징:** 각 미션 셀에서 구현된 단위 모듈과 명세는 **[미션 10]** 실행 시 단일 소프트웨어 패키지(dist/dat06_package/)와 운영 매뉴얼(docs/dat06_final_manual.md)로 일괄 결합되어 자동 추출됩니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 군수 월간 결산 항목(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률) 데이터 구조 분석
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

군수 월간 결산 항목(물자 불출, 수리부속 소모, 탄약 정산) 데이터 구조를 분석한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **월간 결산 4개 항목(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)의 데이터 구조를 분석해 필드와 산식을 정의하고, 현행 결산 업무를 AS-IS/TO-BE로 비교한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **가상 부대 24개의 월간 원장(기초·수령·불출·반납·기말)과 장비 가동 현황을 만들어 항목별 필드와 재고 산식(기말 = 기초 + 수령 + 반납 - 불출) 정리하고 규격화하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/dat06_as_is_to_be_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 현행 업무(AS-IS)와 개선 업무(TO-BE) 비교의 명확성

#### 미션 PASS 조건

- 정량 기준: 업무 단계 분해 10단계 이상, 병목 구간 3개소 이상 도출


- 진행 기준: 업무 정의서 및 입출력 데이터 명세 작성 완료


- 검증 방법: 실무 프로세스와의 일치성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# [TODO : 학생 실습 1] 군수 월간 결산 항목(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률) 데이터 구조 분석
# 미션 지침 및 검증 기준을 확인하고 코드를 실행 및 튜닝하세요.
"""군수 월간 결산 4개 항목(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)의 데이터 구조를 분석해 스키마를 정의하고, 월간 결산 업무를 AS-IS/TO-BE 단계로 분해해 병목 구간을 도출한다."""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 월간 결산 원장 생성기 (실제 부대명·품목·수량·재무 정보 없음)
# 가상 대대 24개 x (물자 불출 / 수리부속 소모 / 탄약 정산) 품목 원장과 장비 가동 현황을 만든다.
# 재고 산식: 기말 = 기초 + 수령 + 반납 - 불출,  다음 달 기초 = 이번 달 기말.
# ---------------------------------------------------------------------------
UNITS = [f"가상{i:02d}대대" for i in range(1, 25)]
ITEMS = {"물자불출": ["급식류", "피복류", "유류", "정비소모품"],
         "수리부속": ["필터류", "베어링류", "전장품", "타이어류"],
         "탄약정산": ["훈련탄A", "훈련탄B", "모의탄"]}
UNIT_CODE = {"급식류": "BX", "피복류": "EA", "유류": "L", "정비소모품": "EA", "필터류": "EA", "베어링류": "EA",
             "전장품": "EA", "타이어류": "EA", "훈련탄A": "RD", "훈련탄B": "RD", "모의탄": "RD"}
LEDGER_COLS = ["month", "unit", "cat", "item", "open", "recv", "issue", "ret", "close"]


def make_ledger(n_months=2, seed=60):
    """n_months개월 원장(list of dict). 둘째 달 기초는 첫째 달 기말과 같다."""
    rng = np.random.default_rng(seed)
    rows, prev = [], {}
    for m in range(n_months):
        for u in UNITS:
            for cat, items in ITEMS.items():
                for it in items:
                    base = int(rng.integers(80, 400))
                    op = prev.get((u, it), base * 3)
                    recv = int(rng.integers(0, base))
                    issue = int(rng.integers(base // 4, base // 2 + 1))
                    ret = int(issue * rng.uniform(0, 0.15))
                    cl = op + recv + ret - issue
                    rows.append({"month": f"2026-{m + 7:02d}", "unit": u, "cat": cat, "item": it,
                                 "open": op, "recv": recv, "issue": issue, "ret": ret, "close": cl})
                    prev[(u, it)] = cl
    return rows


def make_equip(n_months=2, seed=61):
    rng = np.random.default_rng(seed)
    out = []
    for m in range(n_months):
        for u in UNITS:
            hold = int(rng.integers(20, 60))
            out.append({"month": f"2026-{m + 7:02d}", "unit": u, "hold": hold, "operable": int(hold * rng.uniform(0.78, 0.97))})
    return out

# 1) 데이터 구조 분석: 원장 4개 항목의 필드·허용값·산식
led = pd.DataFrame(make_ledger(2), columns=LEDGER_COLS)
eq = pd.DataFrame(make_equip(2))
SCHEMA = {
    "물자불출": {"fields": "month, unit, item, open, recv, issue, ret, close", "rule": "close = open + recv + ret - issue"},
    "수리부속": {"fields": "month, unit, item, open, recv, issue(소모), ret, close", "rule": "close = open + recv + ret - issue"},
    "탄약정산": {"fields": "month, unit, item, open, recv, issue(사용), ret(잔탄 반납), close", "rule": "close = open + recv + ret - issue"},
    "장비가동률": {"fields": "month, unit, hold(보유), operable(가동)", "rule": "가동률 = operable / hold, 0 <= 가동률 <= 1"},
}
print("[결산 항목별 데이터 구조] [교육용가상자료]")
for k, v in SCHEMA.items():
    print(f" - {k}: 필드[{v['fields']}] 규칙[{v['rule']}]")
print(f"\n원장 {len(led)}행(부대 {led['unit'].nunique()}개 x 품목 {led['item'].nunique()}종 x 2개월), 장비 현황 {len(eq)}행")
chk = ((led["open"] + led["recv"] + led["ret"] - led["issue"]) == led["close"]).mean()
cont = led.pivot_table(index=["unit", "item"], columns="month", values=["open", "close"])
cont_ok = (cont[("close", "2026-07")] == cont[("open", "2026-08")]).mean()
print(f"재고 산식 일치율 {chk:.0%}, 월 간 연속성(전월 기말=당월 기초) {cont_ok:.0%}")
print("분류별 월 불출·소모량 합계:")
print(led[led["month"] == "2026-08"].groupby("cat")["issue"].sum().to_string())
print(f"장비 평균 가동률 {(eq['operable'] / eq['hold']).mean():.1%} (가동률 낮은 부대 3개: "
      f"{', '.join(eq.assign(r=eq['operable'] / eq['hold']).nsmallest(3, 'r')['unit'])})")

# 2) 월간 결산 업무 AS-IS 분해(단계, 소요 분 - 가상 측정값)와 TO-BE
steps = [("예하 부대별 결산 자료 수신(메일·메신저)", 40, "제출 양식 표준화·자동 수집 함수", 3),
         ("제출 양식 확인(서식 다른 파일 구분)", 35, "형식 자동 판별·별칭 사전", 2),
         ("엑셀 값 복사·붙여넣기 취합", 90, "자동 파싱·병합", 2),
         ("품목 코드·단위 수기 통일", 60, "코드·단위 표준 변환표", 3),
         ("기초·기말 재고 산식 대조", 55, "검증 룰 15종 자동 적용", 2),
         ("전월 대비 증감 계산", 30, "전월 비교표 자동 생성", 1),
         ("이상 수치 확인(전화 문의)", 75, "이상치 후보·원인 요약 추천", 15),
         ("장비 가동률 집계", 30, "가동률 자동 계산", 1),
         ("결산 보고서 서식 작성", 80, "템플릿에 자동 채움", 3),
         ("보급관·군수과장 결재 요청·회수", 45, "3단계 전자 결재 흐름", 15),
         ("반려 사항 수정 후 재제출", 50, "반려 사유별 수정 위치 안내", 10),
         ("최종본 배포·보관", 20, "발간본 자동 보관·배포", 1)]
proc = pd.DataFrame(steps, columns=["업무 단계", "AS-IS(분)", "TO-BE 개선", "TO-BE(분)"])
proc.index = range(1, len(proc) + 1)
thr = proc["AS-IS(분)"].mean() + 0.5 * proc["AS-IS(분)"].std()
proc["병목"] = np.where(proc["AS-IS(분)"] >= thr, "병목", "")
print(f"\n[AS-IS / TO-BE 업무 단계 {len(proc)}단계] (병목 기준: 평균+0.5표준편차 = {thr:.0f}분 이상)")
print(proc[["업무 단계", "AS-IS(분)", "TO-BE(분)", "병목"]].to_string())
nb = int((proc["병목"] == "병목").sum())
a, t = proc["AS-IS(분)"].sum(), proc["TO-BE(분)"].sum()
print(f"\n업무 단계 {len(proc)}단계(목표 10단계 이상: {'달성' if len(proc) >= 10 else '미달'}), 병목 {nb}개소(목표 3개소 이상: {'달성' if nb >= 3 else '미달'})")
print(f"1회 결산 AS-IS {a}분 -> TO-BE {t}분 ({1 - t / a:.0%} 단축 예상, 가정값)")


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[결산 항목별 데이터 구조] [교육용가상자료]
 - 물자불출: 필드[month, unit, item, open, recv, issue, ret, close] 규칙[close = open + recv + ret - issue]
 - 수리부속: 필드[month, unit, item, open, recv, issue(소모), ret, close] 규칙[close = open + recv + ret - issue]
 - 탄약정산: 필드[month, unit, item, open, recv, issue(사용), ret(잔탄 반납), close] 규칙[close = open + recv + ret - issue]
 - 장비가동률: 필드[month, unit, hold(보유), operable(가동)] 규칙[가동률 = operable / hold, 0 <= 가동률 <= 1]

원장 528행(부대 24개 x 품목 11종 x 2개월), 장비 현황 48행
재고 산식 일치율 100%, 월 간 연속성(전월 기말=당월 기초) 100%
분류별 월 불출·소모량 합계:
cat
물자불출    8270
수리부속    8184
탄약정산    6743
장비 평균 가동률 87.0% (가동률 낮은 부대 3개: 가상23대대, 가상19대대, 가상22대대)

[AS-IS / TO-BE 업무 단계 12단계] (병목 기준: 평균+0.5표준편차 = 62분 이상)
                      업무 단계  AS-IS(분)  TO-BE(분)  병목
1   예하 부대별 결산 자료 수신(메일·메신저)        40         3    
2     제출 양식 확인(서식 다른 파일 구분)        35         2    
3           엑셀 값 복사·붙여넣기 취합        90         2  병목
4            품목 코드·단위 수기 통일        60         3    
5            기초·기말 재고 산식 대조        55         2    
6               전월 대비 증감 계산        30         1    
7           이상 수치 확인(전화 문의)        75        15  병목
8                 장비 가동률 집계        30         1    
9              결산 보고서 서식 작성        80         3  병목
10        보급관·군수과장 결재 요청·회수        45        15    
11           반려 사항 수정 후 재제출        50        10    
12                최종본 배포·보관        20         1    

업무 단계 12단계(목표 10단계 이상: 달성), 병목 3개소(목표 3개소 이상: 달성)
1회 결산 AS-IS 610분 -> TO-BE 58분 (90% 단축 예상, 가정값)
```
</details>
